In [1]:
import pandas as pd
import numpy as np
from scipy import stats
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import classification_report, precision_recall_curve, f1_score, roc_auc_score

In [2]:
files = [
    "/home/truphile/Downloads/classification data/data_descriptions.csv",
    "/home/truphile/Downloads/classification data/train.csv",
    "/home/truphile/Downloads/classification data/test.csv"
]

datasets = [pd.read_csv(file) for file in files]

data, train, test = datasets
data.head()

,Column_name,Column_type,Data_type,Description
0,AccountAge,Feature,integer,The age of the user's account in months.
1,MonthlyCharges,Feature,float,The amount charged to the user on a monthly ba...
2,TotalCharges,Feature,float,The total charges incurred by the user over th...
3,SubscriptionType,Feature,object,The type of subscription chosen by the user (B...
4,PaymentMethod,Feature,string,The method of payment used by the user.


In [3]:
train['TotalCharges'] = pd.to_numeric(train['TotalCharges'], errors='coerce').fillna(0)

In [4]:
train['AvgHistoricalCharge'] = train['TotalCharges'] / train['AccountAge'].replace(0, 1)
train['ChargeIncrease'] = train['MonthlyCharges'] - train['AvgHistoricalCharge']
train['TicketRate'] = train['SupportTicketsPerMonth'] / train['AccountAge'].replace(0, 1)
train['ViewingDurationTotal'] = train['ViewingHoursPerWeek'] * train['AverageViewingDuration']


In [5]:
X = train.drop(columns=['CustomerID', 'Churn'])
y = train['Churn']

In [6]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [7]:
preprocessor = ColumnTransformer([
    ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1),
     make_column_selector(dtype_exclude=np.number))
], remainder='passthrough')

In [8]:
hgb = make_pipeline(
    preprocessor,
    HistGradientBoostingClassifier(
        class_weight='balanced',
        random_state=42,
        early_stopping=True,
        scoring='roc_auc',
        n_iter_no_change=10,
        validation_fraction=0.1
    )
)

In [ ]:
param_distributions = {
    'histgradientboostingclassifier__max_iter': stats.randint(100, 400),          # Equiv to n_estimators
    'histgradientboostingclassifier__max_depth': stats.randint(4, 10),            # Equiv to max_depth
    'histgradientboostingclassifier__min_samples_leaf': stats.randint(10, 50),    # Equiv to min_samples_leaf
    'histgradientboostingclassifier__max_leaf_nodes': stats.randint(15, 60),      # Equiv to min_samples_split (controls splitting)
    'histgradientboostingclassifier__l2_regularization': stats.uniform(0, 2)      # Equiv to max_features (controls feature reliance)
}

random_search = RandomizedSearchCV(
    estimator=hgb,
    param_distributions=param_distributions,
    n_iter=15,                 # 15 combinations for speed
    scoring='f1',
    cv=3,                      # 3 folds for speed
    random_state=42,
    n_jobs=-1,
    verbose=2
)

print("Training HistGradientBoosting with your requested tree structures...")
random_search.fit(X_train, y_train)

print("\nBest Params:", random_search.best_params_)
print("Best Coarse F1 Score:", random_search.best_score_)

Training HistGradientBoosting with your requested tree structures...
Fitting 3 folds for each of 15 candidates, totalling 45 fits
[CV] END histgradientboostingclassifier__l2_regularization=1.1973169683940732, histgradientboostingclassifier__max_depth=5, histgradientboostingclassifier__max_iter=314, histgradientboostingclassifier__max_leaf_nodes=25, histgradientboostingclassifier__min_samples_leaf=20; total time=  10.4s
